In [2]:
!pip install pgmpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]


In [3]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([('T', 'S'), ('S', 'F')])

cpd_t = TabularCPD(
    variable='T',
    variable_card=2,
    values=[[0.7], [0.3]],
    state_names={'T': ['Normal', 'High']}
)

cpd_s = TabularCPD(
    variable='S',
    variable_card=2,
    values=[[0.9, 0.4], [0.1, 0.6]],
    evidence=['T'],
    evidence_card=[2],
    state_names={'S': ['No', 'Yes'], 'T': ['Normal', 'High']}
)

cpd_f = TabularCPD(
    variable='F',
    variable_card=2,
    values=[[0.8, 0.2], [0.2, 0.8]],
    evidence=['S'],
    evidence_card=[2],
    state_names={'F': ['No', 'Yes'], 'S': ['No', 'Yes']}
)

model.add_cpds(cpd_t, cpd_s, cpd_f)

print("Is model valid?", model.check_model())

inference = VariableElimination(model)

print("\nP(Fever | Temperature=High):")
print(inference.query(variables=['F'], evidence={'T': 'High'}))

print("\nP(Fever | Sore throat=Yes):")
print(inference.query(variables=['F'], evidence={'S': 'Yes'}))

print("\nJoint probability of Temperature and Fever:")
print(inference.query(variables=['T', 'F']))

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Is model valid? True

P(Fever | Temperature=High):
+--------+----------+
| F      |   phi(F) |
+========+==========+
| F(No)  |   0.4400 |
+--------+----------+
| F(Yes) |   0.5600 |
+--------+----------+

P(Fever | Sore throat=Yes):
+--------+----------+
| F      |   phi(F) |
+========+==========+
| F(No)  |   0.2000 |
+--------+----------+
| F(Yes) |   0.8000 |
+--------+----------+

Joint probability of Temperature and Fever:
+-----------+--------+------------+
| T         | F      |   phi(T,F) |
+===========+========+============+
| T(Normal) | F(No)  |     0.5180 |
+-----------+--------+------------+
| T(Normal) | F(Yes) |     0.1820 |
+-----------+--------+------------+
| T(High)   | F(No)  |     0.1320 |
+-----------+--------+------------+
| T(High)   | F(Yes) |     0.1680 |
+-----------+--------+------------+
